# Systèmes agentiques composables — Perception → Action

**Notebook** : AgentSystems-01
**Durée** : ~1 h 30
**Prérequis** : .NET (noyau C#), OOP (classes, polymorphisme, types abstraits)

[Version Python](AgentSystems-01-Perception-Vers-Action-Python.ipynb)

## Objectifs d'apprentissage

- Lire la boucle **percevoir → représenter → inférer → décider → vérifier → exécuter → observer** comme un *squelette réutilisable*, pas comme un script.
- Traiter l'**inférence comme un point de composition** : le même agent tourne avec un cerveau déterministe *ou* un LLM, sans que le reste de la boucle change.
- **Réutiliser** les primitives de [`Compression-07b`](../../InformationCalcul/Compression/Compression-07b-Triage-LLM-Semantic-Kernel-Python.ipynb) — la représentation `Segment`, l'invariant de couverture, la boucle de découverte bornée — dans un cadre agentique.
- **Mesurer** l'amortissement du coût : découvrir une politique coûte des inférences, la rejouer en coûte zéro.

> **Position de ce carnet dans le projet.** Il ne ré-invente pas le moteur de
> compression : il en extrait le *squelette d'agent*. Les primitives compression
> sont **réutilisées** (copie pédagogique déclarée — 07b est un notebook, pas un
> module importable) pour montrer la composition ; l'extraction en module
> importable est le pas suivant.

In [1]:
// Préambule — bibliothèque standard .NET
using System;
using System.Collections.Generic;
using System.Globalization;
using System.IO;
using System.IO.Compression;
using System.Linq;
using System.Text;
using System.Text.Encodings.Web;
using System.Text.Json;
using System.Text.RegularExpressions;

// Noyau standard uniquement : ce carnet tourne SANS cle API et SANS GPU.
// Le cerveau LLM (Semantic Kernel) est montre plus bas comme point de composition
// interchangeable ; le resultat experimental ci-dessous est produit par le cerveau
// determine, donc reproductible sur n'importe quelle machine.

void Ecrire(FormattableString texte) => Console.WriteLine(texte.ToString(CultureInfo.InvariantCulture));

string TexteInvariant<T>(T valeur) => string.Format(CultureInfo.InvariantCulture, "{0}", valeur);

// Une unité d'écart désigne une comparaison exacte qui ne coïncide pas.
// Le journal garde les mesures pour l'attestation par section en fin de carnet.
const int ToleranceEgalite = 0;
var comparaisons = new List<(string Section, string Cas, int Ecart)>();

void Egal<T>(string section, string cas, T calcule, T attendu, string origine = "référence Python citée")
{
    int ecart = EqualityComparer<T>.Default.Equals(calcule, attendu) ? 0 : 1;
    comparaisons.Add((section, cas, ecart));
    string verdict = ecart <= ToleranceEgalite ? "CONFORME" : "ÉCART";
    Ecrire($"[{verdict}] {section} / {cas} : C# = {TexteInvariant(calcule)} ; {origine} = {TexteInvariant(attendu)} ; écart = {ecart}");
    if (ecart > ToleranceEgalite)
        throw new InvalidOperationException($"{section} / {cas} : résultat différent de l'attendu.");
}

Console.WriteLine("Imports OK : System.Text.Json, expressions régulières, collections — noyau standard, aucune cle API requise");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Imports OK : System.Text.Json, expressions régulières, collections — noyau standard, aucune cle API requise


## 1. La boucle perception → action

Un système agentique n'est pas un LLM : c'est un **squelette de traitement** que l'on
**compose** avec un mécanisme d'inférence. La boucle de référence a sept stades :

```mermaid
flowchart LR
    P[Percevoir] --> R[Représenter]
    R --> I[Inférer]
    I --> D[Décider]
    D --> V[Vérifier]
    V -->|couverture OK| E[Exécuter]
    V -->|couverture incomplète| I
    E --> O[Observer]
    O -->|politique amortie| R
```

- **Percevoir / Représenter** : ingérer les entrées brutes dans une *représentation commune* (`Segment`).
- **Inférer / Décider** : proposer un traitement — c'est le **point de composition** (le « cerveau »).
- **Vérifier** : un **invariant** rejette tout plan incomplet (un segment resté sans traitement final).
- **Exécuter / Observer** : appliquer le plan **sans ré-inférer**, puis mesurer le coût.

Le point central : **Inférer** est isolé derrière une *interface*. On y branche un
cerveau déterministe, un LLM, ou les deux — le reste de la boucle ne change pas.
C'est cela, être **composable**.

## 2. Représenter — la perception comme signature de structure

La boucle commence par une **représentation commune**. On réutilise la structure
`Segment` de Compression-07b : un morceau de donnée identifié, portant ses octets,
qualifié par un genre, relié à son origine. La *perception* d'un segment, c'est sa
**signature de structure** — déduite du contenu, pas de l'étiquette.

In [2]:
// Perception — construire les cinq segments et déduire leur forme

// Représentation commune, réutilisée de Compression-07b : identifiant stable,
// octets, genre et lien vers l'origine. Une racine a un parent vide.
record Segment(string Id, byte[] Data, string Genre = "brut", string Parent = "", int Profondeur = 0);
record Structure(int Octets, bool Texte, bool Csv, int BlocsMd, bool Json, bool JsonObj);

static class Perception
{
    public static readonly UTF8Encoding Utf8 = new UTF8Encoding(false, true);
    public static readonly Regex EntetesMarkdown = new Regex("^##? ", RegexOptions.Multiline | RegexOptions.CultureInvariant);

    private static readonly JsonSerializerOptions JsonIndente = new JsonSerializerOptions
    {
        WriteIndented = true,
        Encoder = JavaScriptEncoder.UnsafeRelaxedJsonEscaping
    };

    // Le format Python utilise deux espaces d'indentation et des fins de ligne LF.
    // Normaliser les fins de ligne du sérialiseur pour le corpus et le recollage JSON.
    public static string SerialiserJson<T>(T valeur) =>
        JsonSerializer.Serialize(valeur, JsonIndente).Replace("\r\n", "\n", StringComparison.Ordinal);

    // Comme splitlines() suivi du filtre des lignes vides dans le Python.
    public static string[] LignesNonVides(string texte) =>
        Regex.Split(texte, @"\r\n|[\n\r\v\f\x1C-\x1E\u0085\u2028\u2029]")
             .Where(ligne => ligne.Length != 0).ToArray();

    // La structure est déduite du contenu : un nom de fichier ne suffit pas.
    public static Structure StructureInterne(Segment seg)
    {
        int octets = seg.Data.Length;
        string texte;
        try
        {
            texte = Utf8.GetString(seg.Data);
        }
        catch (DecoderFallbackException)
        {
            return new Structure(octets, false, false, 0, false, false);
        }

        string debut = texte.TrimStart();
        if (debut.Length > 0 && (debut[0] == '{' || debut[0] == '['))
        {
            try
            {
                using var document = JsonDocument.Parse(texte);
                bool objet = document.RootElement.ValueKind == JsonValueKind.Object;
                return new Structure(octets, true, false, 0, true, objet);
            }
            catch (JsonException)
            {
                // Le préfixe ne suffit pas : essayer les autres formes textuelles.
            }
        }

        var lignes = LignesNonVides(texte);
        int virgules = lignes.Length == 0 ? 0 : lignes[0].Count(c => c == ',');
        bool csv = virgules > 0 && lignes.All(ligne => ligne.Count(c => c == ',') == virgules);
        return new Structure(octets, true, csv, EntetesMarkdown.Matches(texte).Count, false, false);
    }

    // Distinguer l'objet JSON découpable par clés du tableau JSON compressé tel quel.
    public static string Signature(Structure s)
    {
        if (s.Json) return s.JsonObj ? "json_obj" : "json";
        if (s.Csv) return "csv";
        if (s.BlocsMd > 0) return "markdown";
        return s.Texte ? "texte" : "binaire";
    }

    public static Dictionary<string, Segment> CorpusDemo()
    {
        var ventes = Utf8.GetBytes("nom,quantite,prix\ncafe,10,2.50\nthe,20,1.80\nsucre,5,0.90\n" +
                                   "lait,8,1.10\npain,12,0.80\n");
        var clients = Utf8.GetBytes(SerialiserJson(new
        {
            id = 7,
            panier = new[] { new { sku = "cafe", q = 10 }, new { sku = "the", q = 20 } }
        }) + "\n");
        var notes = Utf8.GetBytes("# Rapport\n\n## Chiffre\n12 kEUR, en hausse.\n\n" +
                                  "## Risque\nFournisseur unique, lead time 6 semaines.\n");
        var journal = Utf8.GetBytes(string.Concat(Enumerable.Range(0, 40).Select(i =>
            FormattableString.Invariant($"10:{i:00}:00 INFO requete {i} ok\n"))));
        var binaire = Enumerable.Range(0, 1024).Select(i => (byte)(i % 256)).ToArray();

        return new Dictionary<string, Segment>(StringComparer.Ordinal)
        {
            ["ventes.csv#0"] = new Segment("ventes.csv#0", ventes, "donnees/csv"),
            ["clients.json#0"] = new Segment("clients.json#0", clients, "donnees/json"),
            ["notes.md#0"] = new Segment("notes.md#0", notes, "texte/markdown"),
            ["journal.log#0"] = new Segment("journal.log#0", journal, "texte/log"),
            ["brut.bin#0"] = new Segment("brut.bin#0", binaire, "binaire")
        };
    }
}

// Sortie Python citée, cellule 4 : une référence par ligne, dans l'ordre affiché.
var referencePerception = new (string Id, int Octets, string Forme, bool Json, bool Csv, int BlocsMd, bool Texte)[]
{
    ("ventes.csv#0", 81, "csv", false, true, 0, true),
    ("clients.json#0", 127, "json_obj", true, false, 0, true),
    ("notes.md#0", 95, "markdown", false, false, 3, true),
    ("journal.log#0", 1110, "texte", false, false, 0, true),
    ("brut.bin#0", 1024, "binaire", false, false, 0, false)
};

// Démonstration de la perception : pour chaque segment du corpus, on lit sa forme
// interne déduite du contenu (StructureInterne) puis sa classe de forme
// (Signature) — sans aucune étiquette, seulement à partir des octets.
var segments = Perception.CorpusDemo();
var lignesPercues = segments.Select(entree =>
{
    var structure = Perception.StructureInterne(entree.Value);
    return (Id: entree.Value.Id, structure.Octets, Forme: Perception.Signature(structure),
            structure.Json, structure.Csv, structure.BlocsMd, structure.Texte);
}).ToArray();
int formesDistinctes = lignesPercues.Select(ligne => ligne.Forme).Distinct(StringComparer.Ordinal).Count();

foreach (var ligne in lignesPercues)
    Ecrire($"  {ligne.Id,-16} {ligne.Octets,5} o   forme={ligne.Forme,-9} json={ligne.Json} csv={ligne.Csv} blocs_md={ligne.BlocsMd} texte={ligne.Texte}");
Ecrire($"\ncorpus percu : {segments.Count} segments heterogenes, {formesDistinctes} formes distinctes");

Egal("Perception", "nombre de segments", segments.Count, 5);
Egal("Perception", "nombre de formes distinctes", formesDistinctes, 5);
for (int i = 0; i < referencePerception.Length; i++)
{
    var mesure = lignesPercues[i];
    var attendu = referencePerception[i];
    Egal("Perception", $"{attendu.Id} : identifiant, octets, forme et indicateurs", mesure, attendu);
}

  ventes.csv#0        81 o   forme=csv       json=False csv=True blocs_md=0 texte=True


  clients.json#0     127 o   forme=json_obj  json=True csv=False blocs_md=0 texte=True


  notes.md#0          95 o   forme=markdown  json=False csv=False blocs_md=3 texte=True


  journal.log#0     1110 o   forme=texte     json=False csv=False blocs_md=0 texte=True


  brut.bin#0        1024 o   forme=binaire   json=False csv=False blocs_md=0 texte=False



corpus percu : 5 segments heterogenes, 5 formes distinctes


... (3 outputs left out)

[CONFORME] Perception / clients.json#0 : identifiant, octets, forme et indicateurs : C# = (clients.json#0, 127, json_obj, True, False, 0, True) ; référence Python citée = (clients.json#0, 127, json_obj, True, False, 0, True) ; écart = 0


[CONFORME] Perception / notes.md#0 : identifiant, octets, forme et indicateurs : C# = (notes.md#0, 95, markdown, False, False, 3, True) ; référence Python citée = (notes.md#0, 95, markdown, False, False, 3, True) ; écart = 0


[CONFORME] Perception / journal.log#0 : identifiant, octets, forme et indicateurs : C# = (journal.log#0, 1110, texte, False, False, 0, True) ; référence Python citée = (journal.log#0, 1110, texte, False, False, 0, True) ; écart = 0


[CONFORME] Perception / brut.bin#0 : identifiant, octets, forme et indicateurs : C# = (brut.bin#0, 1024, binaire, False, False, 0, False) ; référence Python citée = (brut.bin#0, 1024, binaire, False, False, 0, False) ; écart = 0


### Lecture du résultat

Cinq segments, **cinq formes distinctes** : le CSV et le JSON sont des structures
*exploitables* (on peut les découper en parties de nature différente), le markdown
est structuré par ses en-têtes, le journal est un texte plat, le binaire n'a pas de
structure lisible. C'est exactement la hétérogénéité que la boucle doit gérer — et
c'est ce qui rendra les décisions **non uniformes** plus loin (le tri n'est pas
trivial : chaque forme appelle une opération différente).

## 3. Inférer — le point de composition

Le stade **Inférer** est isolé derrière une *interface* : `Decision Cerveau.Inferer(Segment seg)`.
C'est ce contrat unique — et rien d'autre — qui rend le cerveau
**interchangeable**. La boucle ne sait pas *qui* infère, seulement *que* quelqu'un
infère. On commence par le cerveau **déterministe**, qui dérive la décision de la
signature perçue : c'est lui qui produira le résultat expérimental.

In [3]:
// Inférence — politique déterministe et aperçu

// Le paramètre désigne le mode de découpe ou le codec selon l'opération.
record Decision(string Op, string Parametre);

// Le contrat est commun aux cerveaux déterministe, LLM et hybride.
abstract class Cerveau
{
    public int Appels { get; protected set; }
    public abstract Decision Inferer(Segment seg);
}

sealed class CerveauDeterministe : Cerveau
{
    public const int SeuilDecoupe = 64; // un segment trop petit n'a pas d'interet a etre decoupe

    public override Decision Inferer(Segment seg)
    {
        Appels++;
        var structure = Perception.StructureInterne(seg);
        var compression = new Decision("compresser", structure.Texte ? "zlib" : "brotli");
        if (structure.Octets < SeuilDecoupe)
            return compression;

        return Perception.Signature(structure) switch
        {
            "json_obj" => new Decision("decouper", "cles_json"),
            "csv" => new Decision("decouper", "colonnes_csv"),
            "markdown" => new Decision("decouper", "blocs_markdown"),
            _ => compression
        };
    }
}

// Correspondance de comparaison uniquement : la politique et la recette C#
// gardent le nom du codec réellement utilisé. Les flux Brotli et LZMA diffèrent.
Decision VersReferencePython(Decision decision) =>
    decision.Op == "compresser" && decision.Parametre == "brotli"
        ? new Decision(decision.Op, "lzma")
        : decision;

// Sortie Python citée, cellule 7 : décisions dans l'ordre de l'aperçu.
var referenceApercu = new (string Id, Decision Decision)[]
{
    ("ventes.csv#0", new Decision("decouper", "colonnes_csv")),
    ("clients.json#0", new Decision("decouper", "cles_json")),
    ("notes.md#0", new Decision("decouper", "blocs_markdown")),
    ("journal.log#0", new Decision("compresser", "zlib")),
    ("brut.bin#0", new Decision("compresser", "lzma"))
};

var apercu = new CerveauDeterministe();
string nomCerveau = apercu.GetType().Name;
int seuilApercu = CerveauDeterministe.SeuilDecoupe;
Ecrire($"cerveau determine par : {nomCerveau}  (seuil decoupe = {seuilApercu} o)\n");
Console.WriteLine("decision derivee de la forme, segment par segment :");
var decisionsApercu = segments.Select(entree => (Id: entree.Key, Decision: apercu.Inferer(entree.Value))).ToArray();
foreach (var ligne in decisionsApercu)
    Ecrire($"  {ligne.Id,-16} -> {ligne.Decision.Op,-10} {ligne.Decision.Parametre}");
Ecrire($"\ninférences effectuees pour cet apercu : {apercu.Appels}");

Egal("Aperçu", "type du cerveau", nomCerveau, "CerveauDeterministe");
Egal("Aperçu", "seuil de découpe en octets", seuilApercu, 64);
Egal("Aperçu", "nombre de décisions", decisionsApercu.Length, referenceApercu.Length);
for (int i = 0; i < referenceApercu.Length; i++)
{
    var mesure = decisionsApercu[i];
    var attendu = referenceApercu[i];
    string cas = attendu.Decision.Parametre == "lzma"
        ? $"{attendu.Id} : décision avec correspondance Brotli → LZMA"
        : $"{attendu.Id} : décision";
    Egal("Aperçu", cas, (mesure.Id, VersReferencePython(mesure.Decision)), attendu);
}
Egal("Aperçu", "codec binaire natif", decisionsApercu.Single(ligne => ligne.Id == "brut.bin#0").Decision.Parametre,
     "brotli", "codec BCL déclaré");
Egal("Aperçu", "inférences effectuées", apercu.Appels, 5);

cerveau determine par : CerveauDeterministe  (seuil decoupe = 64 o)



decision derivee de la forme, segment par segment :


  ventes.csv#0     -> decouper   colonnes_csv


  clients.json#0   -> decouper   cles_json


  notes.md#0       -> decouper   blocs_markdown


  journal.log#0    -> compresser zlib


... (8 outputs left out)

[CONFORME] Aperçu / journal.log#0 : décision : C# = (journal.log#0, Decision { Op = compresser, Parametre = zlib }) ; référence Python citée = (journal.log#0, Decision { Op = compresser, Parametre = zlib }) ; écart = 0


[CONFORME] Aperçu / brut.bin#0 : décision avec correspondance Brotli → LZMA : C# = (brut.bin#0, Decision { Op = compresser, Parametre = lzma }) ; référence Python citée = (brut.bin#0, Decision { Op = compresser, Parametre = lzma }) ; écart = 0


[CONFORME] Aperçu / codec binaire natif : C# = brotli ; codec BCL déclaré = brotli ; écart = 0


[CONFORME] Aperçu / inférences effectuées : C# = 5 ; référence Python citée = 5 ; écart = 0


### Lecture du résultat

Le cerveau **n'est pas un classifieur en dur** : il *dérive* chaque décision de la
signature perçue. Les formes exploitables (`csv`, `json`, `markdown`) au-delà du
seuil sont **découpées** (l'opération qui *réduit*), le reste est **compressé**
(avec un codec choisi par la nature, texte/binaire). Cinq segments, cinq décisions —
la non-uniformité est là. Et chaque décision a coûté **une inférence** : c'est le
compteur que la boucle va amortir.

## 4. Vérifier — l'invariant de couverture

Le stade **Vérifier** est le garde-fou de la composition : un plan n'est **pas
exécutable** tant qu'il laisse un segment d'origine *sans traitement final*. On
réutilise l'invariant de couverture de Compression-07b (`CouvertureIncomplete` +
`Couverture.VerifierCouverture`) : un segment est couvert s'il est compressé, **ou** s'il est
découpé en enfants *tous* couverts. C'est le **témoin négatif** de l'organe — il
nomme précisément ce qui manque.

In [4]:
// Couverture, découpes et codecs inversibles

sealed class CouvertureIncomplete : InvalidOperationException
{
    public CouvertureIncomplete(string message) : base(message) { }
}

record EntreeRecette(Decision Decision, IReadOnlyList<string> Enfants)
{
    public EntreeRecette(Decision decision) : this(decision, Array.Empty<string>()) { }
}

static class Couverture
{
    // Le chemin courant détecte les cycles sans confondre deux branches.
    private static void RechercherManquants(string id,
        IReadOnlyDictionary<string, EntreeRecette> recette, HashSet<string> chemin,
        List<string> manquants)
    {
        if (!chemin.Add(id))
            throw new CouvertureIncomplete($"cycle dans la recette autour de {id}");

        try
        {
            if (!recette.TryGetValue(id, out var entree))
            {
                manquants.Add(id);
                return;
            }
            if (entree.Decision.Op == "compresser")
                return;
            if (entree.Decision.Op != "decouper" || entree.Enfants.Count == 0)
            {
                manquants.Add(id);
                return;
            }
            foreach (string enfant in entree.Enfants)
                RechercherManquants(enfant, recette, chemin, manquants);
        }
        finally
        {
            chemin.Remove(id);
        }
    }

    public static void VerifierCouverture(IEnumerable<string> racines,
        IReadOnlyDictionary<string, EntreeRecette> recette)
    {
        var manquants = new List<string>();
        var chemin = new HashSet<string>(StringComparer.Ordinal);
        foreach (string id in racines)
            RechercherManquants(id, recette, chemin, manquants);
        if (manquants.Count > 0)
        {
            string extraits = string.Join(", ", manquants.Take(6).Select(id => $"'{id}'"));
            throw new CouvertureIncomplete(FormattableString.Invariant(
                $"{manquants.Count} segment(s) sans traitement final : [{extraits}]"));
        }
    }
}

// Les operations inversibles (réutilisées de 07b) : decouper / recoller, et codecs.
static class OperationsTriage
{
    // ZLibStream porte zlib ; BrotliStream tient le rôle du codec binaire LZMA.
    public static readonly Dictionary<string, (Func<byte[], byte[]> Compresser, Func<byte[], byte[]> Decompresser)> Codecs =
        new Dictionary<string, (Func<byte[], byte[]> Compresser, Func<byte[], byte[]> Decompresser)>(StringComparer.Ordinal)
        {
            ["zlib"] = (d => CompresserAvec(d, flux => new ZLibStream(flux, CompressionLevel.SmallestSize, leaveOpen: true)),
                        d => DecompresserAvec(d, flux => new ZLibStream(flux, CompressionMode.Decompress))),
            ["brotli"] = (d => CompresserAvec(d, flux => new BrotliStream(flux, CompressionLevel.SmallestSize, leaveOpen: true)),
                          d => DecompresserAvec(d, flux => new BrotliStream(flux, CompressionMode.Decompress))),
            ["store"] = (d => d, d => d)
        };

    private static byte[] CompresserAvec(byte[] donnees, Func<Stream, Stream> ouvrir)
    {
        using var sortie = new MemoryStream();
        // Fermer le flux de compression termine l'écriture avant la lecture des octets.
        using (var codec = ouvrir(sortie))
            codec.Write(donnees, 0, donnees.Length);
        return sortie.ToArray();
    }

    private static byte[] DecompresserAvec(byte[] donnees, Func<Stream, Stream> ouvrir)
    {
        using var entree = new MemoryStream(donnees, writable: false);
        using var codec = ouvrir(entree);
        using var sortie = new MemoryStream();
        codec.CopyTo(sortie);
        return sortie.ToArray();
    }

    public static List<Segment> Decouper(Segment seg, string mode)
    {
        string texte = Perception.Utf8.GetString(seg.Data); // les modes ci-dessous ne s'appliquent qu'au texte
        var enfants = new List<Segment>();

        Segment Enfant(string prefixe, int index, string contenu, string genre) =>
            new Segment(FormattableString.Invariant($"{seg.Id}.{prefixe}{index}"),
                Perception.Utf8.GetBytes(contenu), $"{seg.Genre}/{genre}", seg.Id, seg.Profondeur + 1);

        switch (mode)
        {
            case "blocs_markdown":
            {
                var bornes = new List<int> { 0 };
                foreach (Match entete in Perception.EntetesMarkdown.Matches(texte))
                    bornes.Add(entete.Index);
                bornes.Add(texte.Length);
                // L'indice est celui du morceau avant le filtre, comme dans le Python.
                for (int i = 0; i < bornes.Count - 1; i++)
                {
                    string morceau = texte.Substring(bornes[i], bornes[i + 1] - bornes[i]);
                    if (!string.IsNullOrWhiteSpace(morceau))
                        enfants.Add(Enfant("b", i, morceau, "bloc"));
                }
                break;
            }
            case "colonnes_csv":
            {
                var matrice = Perception.LignesNonVides(texte).Select(ligne => ligne.Split(',')).ToArray();
                var colonnes = matrice[0];
                for (int j = 0; j < colonnes.Length; j++)
                {
                    string contenu = string.Join("\n", matrice.Select(ligne => ligne[j]));
                    enfants.Add(Enfant("c", j, contenu, $"col:{colonnes[j]}"));
                }
                break;
            }
            case "cles_json":
            {
                using var document = JsonDocument.Parse(texte);
                int i = 0;
                foreach (var propriete in document.RootElement.EnumerateObject())
                    enfants.Add(Enfant("k", i++, Perception.SerialiserJson(propriete.Value), $"cle:{propriete.Name}"));
                break;
            }
            default:
                throw new ArgumentException($"mode de decoupe inconnu : {mode}", nameof(mode));
        }
        return enfants;
    }

    public static byte[] Recoller(string mode, IReadOnlyList<Segment> enfants)
    {
        switch (mode)
        {
            case "blocs_markdown":
                return enfants.SelectMany(enfant => enfant.Data).ToArray();
            case "colonnes_csv":
            {
                var colonnes = enfants.Select(enfant => Perception.Utf8.GetString(enfant.Data).Split('\n')).ToArray();
                var lignes = Enumerable.Range(0, colonnes[0].Length)
                    .Select(i => string.Join(",", colonnes.Select(colonne => colonne[i])));
                return Perception.Utf8.GetBytes(string.Join("\n", lignes) + "\n");
            }
            case "cles_json":
            {
                var objet = new Dictionary<string, JsonElement>(StringComparer.Ordinal);
                foreach (var enfant in enfants)
                {
                    string cle = enfant.Genre.Substring(enfant.Genre.LastIndexOf(':') + 1);
                    using var document = JsonDocument.Parse(enfant.Data);
                    objet[cle] = document.RootElement.Clone();
                }
                return Perception.Utf8.GetBytes(Perception.SerialiserJson(objet) + "\n");
            }
            default:
                throw new ArgumentException($"mode de recolle inconnu : {mode}", nameof(mode));
        }
    }
}

// Contrat d'inversibilite (temoin de l'organe) : decouper puis recoller redonne
// les octets d'origine, pour chaque mode present dans le corpus.
const string sectionOperations = "Opérations inversibles";
var modesDecoupe = new[] { "blocs_markdown", "colonnes_csv", "cles_json" };
var modesVerifies = new HashSet<string>(StringComparer.Ordinal);
foreach (var segment in segments.Values)
{
    var structure = Perception.StructureInterne(segment);
    foreach (string mode in modesDecoupe)
    {
        bool applicable = mode switch
        {
            "blocs_markdown" => structure.BlocsMd > 0,
            "colonnes_csv" => structure.Csv,
            "cles_json" => structure.JsonObj,
            _ => false
        };
        if (!applicable)
            continue; // mode inapplicable a ce segment : on passe au suivant

        var enfants = OperationsTriage.Decouper(segment, mode);
        Egal(sectionOperations, $"{segment.Id} / {mode} : découpe non vide", enfants.Count > 0,
            true, "contrat de découpe");
        var reconstruit = OperationsTriage.Recoller(mode, enfants);
        Egal(sectionOperations, $"{segment.Id} / {mode} : égalité octet pour octet",
            reconstruit.SequenceEqual(segment.Data), true, "invariant Python");
        modesVerifies.Add(mode);
    }
}
Egal(sectionOperations, "tous les modes ont été vérifiés", modesVerifies.SetEquals(modesDecoupe),
    true, "contrat d'inversibilité sur le corpus");

string ListeCodecs(IEnumerable<string> noms) =>
    "[" + string.Join(", ", noms.OrderBy(nom => nom, StringComparer.Ordinal).Select(nom => $"'{nom}'")) + "]";

// Liste de la sortie Python (cellule 10), citée et comparée après correspondance.
const string referenceCodecsPython = "['lzma', 'store', 'zlib']";
string codecsBcl = ListeCodecs(OperationsTriage.Codecs.Keys);
Egal(sectionOperations, "liste native", codecsBcl, "['brotli', 'store', 'zlib']", "registre BCL déclaré");
string codecsCorrespondants = ListeCodecs(OperationsTriage.Codecs.Keys.Select(nom =>
    VersReferencePython(new Decision("compresser", nom)).Parametre));
Egal(sectionOperations, "liste avec correspondance Brotli → LZMA", codecsCorrespondants, referenceCodecsPython);
Ecrire($"codecs : {codecsBcl} ; decoupes/recolles verifiés inversibles sur le corpus");
Ecrire($"codecs Python (liste citée) : {referenceCodecsPython}");

byte[] temoinCodec = Perception.Utf8.GetBytes(string.Concat(Enumerable.Repeat("temoin de codec ", 5)));
var entreesCodecs = segments.Values.Select(seg => (Id: seg.Id, Donnees: seg.Data))
    .Prepend((Id: "témoin de codec", Donnees: temoinCodec));
foreach (var codec in OperationsTriage.Codecs)
{
    foreach (var entree in entreesCodecs)
    {
        var reconstruit = codec.Value.Decompresser(codec.Value.Compresser(entree.Donnees));
        Egal(sectionOperations, $"{codec.Key} / {entree.Id} : égalité octet pour octet",
            reconstruit.SequenceEqual(entree.Donnees), true, "invariant Python");
    }
}
Console.WriteLine("codecs inversibles : OK");

[CONFORME] Opérations inversibles / ventes.csv#0 / colonnes_csv : découpe non vide : C# = True ; contrat de découpe = True ; écart = 0


[CONFORME] Opérations inversibles / ventes.csv#0 / colonnes_csv : égalité octet pour octet : C# = True ; invariant Python = True ; écart = 0


[CONFORME] Opérations inversibles / clients.json#0 / cles_json : découpe non vide : C# = True ; contrat de découpe = True ; écart = 0


[CONFORME] Opérations inversibles / clients.json#0 / cles_json : égalité octet pour octet : C# = True ; invariant Python = True ; écart = 0


[CONFORME] Opérations inversibles / notes.md#0 / blocs_markdown : découpe non vide : C# = True ; contrat de découpe = True ; écart = 0


[CONFORME] Opérations inversibles / notes.md#0 / blocs_markdown : égalité octet pour octet : C# = True ; invariant Python = True ; écart = 0


... (20 outputs left out)

[CONFORME] Opérations inversibles / store / notes.md#0 : égalité octet pour octet : C# = True ; invariant Python = True ; écart = 0


[CONFORME] Opérations inversibles / store / journal.log#0 : égalité octet pour octet : C# = True ; invariant Python = True ; écart = 0


[CONFORME] Opérations inversibles / store / brut.bin#0 : égalité octet pour octet : C# = True ; invariant Python = True ; écart = 0


codecs inversibles : OK


### Lecture du résultat

Deux familles d'opérations, toutes **inversibles** sur le corpus : les
*découpes* (csv/json/markdown) et les *codecs* (`zlib` avec `ZLibStream`, `brotli`
avec `BrotliStream` à la place de `lzma`, et `store`). L'invariant de
couverture qui vient d'être défini est le **garde-fou de la composition** : il sera
déclenché plus bas sur un plan *délibérément* incomplet, pour montrer que la boucle
**rejette** un plan qu'elle ne peut pas exécuter en toute sécurité.

> **Note de parité des codecs.** La liste Python est citée ; le contrat vérifié
> porte sur les octets restaurés, pas sur l'identité des flux Brotli et LZMA.

## 5. Le squelette — `percevoir → … → observer`

On assemble maintenant les stades en un **squelette** `AgentTriage`. Il compose :
la représentation (`Segment`), un cerveau (le point de composition), l'invariant de
couverture (la vérification), et l'application de la recette (l'exécution). Le
résultat clé est dans `Decouvrir` / `Rejouer` : la découverte **paie** le
catalogue `forme → décision` en inférences ; le rejeu le **consomme** sans
ré-inférer.

In [5]:
// AgentTriage — découverte bornée et rejeu du catalogue

sealed class BorneDecouverteAtteinte : InvalidOperationException
{
    public int Inferences { get; }
    public int EnAttente { get; }

    public BorneDecouverteAtteinte(int inferences, int enAttente)
        : base(FormattableString.Invariant(
            $"decouverte interrompue a {inferences} inférences, {enAttente} segment(s) en attente — safety bound."))
    {
        Inferences = inferences;
        EnAttente = enAttente;
    }
}

sealed class AgentTriage
{
    public Cerveau Cerveau { get; }
    public int Borne { get; }
    private readonly Dictionary<string, Decision> catalogue = new Dictionary<string, Decision>(StringComparer.Ordinal); // forme -> decision  (le savoir amorti)
    private readonly Dictionary<string, EntreeRecette> recette = new Dictionary<string, EntreeRecette>(StringComparer.Ordinal); // id segment -> decision

    public IReadOnlyDictionary<string, Decision> Catalogue => catalogue;
    public IReadOnlyDictionary<string, EntreeRecette> Recette => recette;
    public int AppelDecouverte { get; private set; }

    public AgentTriage(Cerveau cerveau, int borne = 64)
    {
        Cerveau = cerveau ?? throw new ArgumentNullException(nameof(cerveau));
        Borne = borne;
    }

    // percevoir + representer : l'entree est deja des Segment (perception faite)
    public IReadOnlyDictionary<string, EntreeRecette> Decouvrir(IEnumerable<Segment> segments)
    {
        catalogue.Clear();
        recette.Clear();
        AppelDecouverte = 0;
        var racines = segments.ToArray();
        var file = new Queue<Segment>(racines);
        while (file.Count > 0)
        {
            if (AppelDecouverte >= Borne)
                throw new BorneDecouverteAtteinte(AppelDecouverte, file.Count);

            var seg = file.Dequeue();
            var decision = Cerveau.Inferer(seg); // inferer + decider (compté)
            AppelDecouverte++;
            catalogue[Perception.Signature(Perception.StructureInterne(seg))] = decision;
            if (decision.Op == "decouper")
            {
                var enfants = OperationsTriage.Decouper(seg, decision.Parametre);
                // la recette porte les ENFANTS reels, pas seulement la decision :
                // c'est cette liste que l'invariant de couverture parcourt.
                recette[seg.Id] = new EntreeRecette(decision, enfants.Select(enfant => enfant.Id).ToArray());
                foreach (var enfant in enfants)
                    file.Enqueue(enfant);
            }
            else
            {
                recette[seg.Id] = new EntreeRecette(decision);
            }
        }
        Couverture.VerifierCouverture(racines.Select(seg => seg.Id), recette); // verifier
        return Recette;
    }

    // Le rejeu consomme le catalogue : il ne consulte pas le cerveau.
    public Dictionary<string, Decision> Rejouer(IEnumerable<Segment> segments)
    {
        var plan = new Dictionary<string, Decision>(StringComparer.Ordinal);
        foreach (var seg in segments)
            plan[seg.Id] = catalogue[Perception.Signature(Perception.StructureInterne(seg))];
        return plan;
    }
}

### Lecture du résultat

Le squelette tient en une classe. Notez la place exacte de chaque stade : `Decouvrir`
fait **percevoir→représenter→inférer→décider→vérifier** d'un bloc (la vérification
est *dans* la découverte — un plan incomplet n'est jamais accepté), et `Rejouer`
fait **exécuter** le savoir acquis **sans passer par l'inférence**. `observer`,
c'est le compteur `AppelDecouverte` qu'on va lire dans la section suivante.

## 6. Exécuter + Observer — l'amortissement du coût

On laisse le squelette **découvrir** la politique sur le corpus, puis on le fait
**rejouer** sur un *deuxième* corpus de même forme. L'observation est le contraste
de coût entre les deux phases.

In [6]:
// Découverte — construire le catalogue et la recette

// Découverte : on instancie l'agent (cerveau déterministe, borne de 64 appels), on
// lui fait percevoir le corpus, puis on lit le catalogue amorti de formes et la
// recette — ce que l'agent a déduit de ce qu'il sait faire pour chaque forme.
var agent = new AgentTriage(new CerveauDeterministe(), borne: 64);
var racinesDecouverte = segments.Values.ToArray();
var recetteDecouverte = agent.Decouvrir(racinesDecouverte);
var catalogueDecouvert = agent.Catalogue.ToArray();

// Sortie Python citée, cellule 16 : catalogue final dans l'ordre affiché.
var referenceCatalogue = new (string Forme, Decision Decision)[]
{
    ("csv", new Decision("decouper", "colonnes_csv")),
    ("json_obj", new Decision("decouper", "cles_json")),
    ("markdown", new Decision("compresser", "zlib")),
    ("texte", new Decision("compresser", "zlib")),
    ("binaire", new Decision("compresser", "lzma")),
    ("json", new Decision("compresser", "zlib"))
};

Ecrire($"DECOUVERTE (1er corpus, {racinesDecouverte.Length} racines) :");
Ecrire($"   inférences effectuees : {agent.AppelDecouverte}");
Ecrire($"   catalogue amorti ({catalogueDecouvert.Length} formes) :");
foreach (var entree in catalogueDecouvert)
    Ecrire($"      {entree.Key,-9} -> {entree.Value.Op,-10} {entree.Value.Parametre}");
Ecrire($"   recette (segments couverts) : {recetteDecouverte.Count} entrees");

const string sectionDecouverte = "Découverte";
Egal(sectionDecouverte, "racines", racinesDecouverte.Length, 5);
Egal(sectionDecouverte, "inférences", agent.AppelDecouverte, 13);
Egal(sectionDecouverte, "appels réels du cerveau", agent.Cerveau.Appels, agent.AppelDecouverte, "compteur de découverte");
Egal(sectionDecouverte, "formes du catalogue", catalogueDecouvert.Length, referenceCatalogue.Length);
Egal(sectionDecouverte, "entrées de recette", recetteDecouverte.Count, 13);
for (int i = 0; i < referenceCatalogue.Length; i++)
{
    var mesure = catalogueDecouvert[i];
    var attendu = referenceCatalogue[i];
    string cas = attendu.Forme == "binaire" ? "binaire : correspondance Brotli → LZMA" : attendu.Forme;
    Egal(sectionDecouverte, cas, (mesure.Key, VersReferencePython(mesure.Value)), attendu);
}
Egal(sectionDecouverte, "codec binaire natif", agent.Catalogue["binaire"].Parametre, "brotli", "codec BCL déclaré");

DECOUVERTE (1er corpus, 5 racines) :


   inférences effectuees : 13


   catalogue amorti (6 formes) :


      csv       -> decouper   colonnes_csv


      json_obj  -> decouper   cles_json


      markdown  -> compresser zlib


... (12 outputs left out)

[CONFORME] Découverte / texte : C# = (texte, Decision { Op = compresser, Parametre = zlib }) ; référence Python citée = (texte, Decision { Op = compresser, Parametre = zlib }) ; écart = 0


[CONFORME] Découverte / binaire : correspondance Brotli → LZMA : C# = (binaire, Decision { Op = compresser, Parametre = lzma }) ; référence Python citée = (binaire, Decision { Op = compresser, Parametre = lzma }) ; écart = 0


[CONFORME] Découverte / json : C# = (json, Decision { Op = compresser, Parametre = zlib }) ; référence Python citée = (json, Decision { Op = compresser, Parametre = zlib }) ; écart = 0


[CONFORME] Découverte / codec binaire natif : C# = brotli ; codec BCL déclaré = brotli ; écart = 0


In [7]:
// Rejeu — nouvelles ventes, aucun appel au cerveau

// Un DEUXIEME corpus, meme forme, donnees differentes : le rejeu consomme le
// catalogue appris, sans re-inferer une seule fois.
var ventes2 = Perception.Utf8.GetBytes("nom,quantite,prix\ncafe,4,2.50\nbiere,16,3.00\n");
var nouveau = new[] { new Segment("ventes2.csv#0", ventes2, "donnees/csv") };
int appelsAvantRejeu = agent.Cerveau.Appels;
var planRejeu = agent.Rejouer(nouveau);
int inferencesRejeu = agent.Cerveau.Appels - appelsAvantRejeu;

Ecrire($"REJEU (2e corpus, {nouveau.Length} racine) :");
foreach (var entree in planRejeu)
    Ecrire($"   {entree.Key,-16} -> {entree.Value.Op,-10} {entree.Value.Parametre}   ({inferencesRejeu} inférence)");
Ecrire($"   inférences effectuees : {inferencesRejeu}  (le catalogue a ete consulte, pas le cerveau)");

// Sortie Python citée, cellule 17 : une racine, une décision et zéro inférence.
Egal("Rejeu", "racines", nouveau.Length, 1);
Egal("Rejeu", "entrées du plan", planRejeu.Count, 1);
var decisionRejouee = planRejeu.Single();
Egal("Rejeu", "identifiant et décision", (decisionRejouee.Key, decisionRejouee.Value),
    ("ventes2.csv#0", new Decision("decouper", "colonnes_csv")));
Egal("Rejeu", "delta du compteur du cerveau", inferencesRejeu, 0);

REJEU (2e corpus, 1 racine) :


   ventes2.csv#0    -> decouper   colonnes_csv   (0 inférence)


   inférences effectuees : 0  (le catalogue a ete consulte, pas le cerveau)


[CONFORME] Rejeu / racines : C# = 1 ; référence Python citée = 1 ; écart = 0


[CONFORME] Rejeu / entrées du plan : C# = 1 ; référence Python citée = 1 ; écart = 0


[CONFORME] Rejeu / identifiant et décision : C# = (ventes2.csv#0, Decision { Op = decouper, Parametre = colonnes_csv }) ; référence Python citée = (ventes2.csv#0, Decision { Op = decouper, Parametre = colonnes_csv }) ; écart = 0


[CONFORME] Rejeu / delta du compteur du cerveau : C# = 0 ; référence Python citée = 0 ; écart = 0


In [8]:
// Observation — comparer les coûts d'inférence

// L'observation : le cout d'inference est PAYE une fois (decouverte) puis amorti
// a zero (rejeu). C'est la these de Compression-07b, reformulee comme propriete
// d'un agent composable : la connaissance forme->decision est un ACTIF amortissable.
int coutDecouverte = agent.AppelDecouverte;
int coutRejeu = inferencesRejeu;

// Projection, pas une mesure sur 1000 corpus : le premier sert à la découverte.
const int nombreCorpus = 1000;
int coutAvecCatalogue = coutDecouverte + (nombreCorpus - 1) * coutRejeu;
int coutSansCatalogue = nombreCorpus * coutDecouverte;

Console.WriteLine("OBSERVATION — amortissement du cout d'inference :");
Ecrire($"   decouverte : {coutDecouverte} inférences   (politique apprise)");
Ecrire($"   rejeu      : {coutRejeu} inférences       (politique consommee)");
Ecrire($"   -> sur {nombreCorpus} corpus de meme forme, on re-infererait {coutAvecCatalogue} fois, pas {nombreCorpus} x {coutDecouverte}.");

// Références citées de la sortie Python, cellule 18, y compris son produit indiqué.
Egal("Observation", "coût de la découverte", coutDecouverte, 13);
Egal("Observation", "coût du rejeu", coutRejeu, 0);
Egal("Observation", "nombre de corpus dans la projection", nombreCorpus, 1000);
Egal("Observation", "coût projeté avec catalogue", coutAvecCatalogue, 13);
Egal("Observation", "coût projeté sans catalogue", coutSansCatalogue, 1000 * 13);

OBSERVATION — amortissement du cout d'inference :


   decouverte : 13 inférences   (politique apprise)


   rejeu      : 0 inférences       (politique consommee)


   -> sur 1000 corpus de meme forme, on re-infererait 13 fois, pas 1000 x 13.


[CONFORME] Observation / coût de la découverte : C# = 13 ; référence Python citée = 13 ; écart = 0


[CONFORME] Observation / coût du rejeu : C# = 0 ; référence Python citée = 0 ; écart = 0


[CONFORME] Observation / nombre de corpus dans la projection : C# = 1000 ; référence Python citée = 1000 ; écart = 0


[CONFORME] Observation / coût projeté avec catalogue : C# = 13 ; référence Python citée = 13 ; écart = 0


[CONFORME] Observation / coût projeté sans catalogue : C# = 13000 ; référence Python citée = 13000 ; écart = 0


### Lecture du résultat

C'est le **résultat expérimental** du carnet : la découverte a coûté
`13` inférences pour apprendre le catalogue
`forme → décision` ; le rejeu sur un corpus *nouveau* de même forme a coûté **zéro**
inférence, car elle a consulté le catalogue, pas le cerveau. L'amortissement est
réel et mesurable : la connaissance est un actif qui se paie une fois. C'est ce que
« composition » veut dire concrètement — le coût d'inférence est *déplacé* de
l'exécution vers la découverte, et l'exécution devient gratuite en inférences.

## 7. La vérification en action — rejeter un plan incomplet

Le garde-fou n'est pas théorique : on le **déclenche**. On construit un plan où un
segment est découpé mais dont *un enfant* est laissé sans traitement final, et on
le passe à `Couverture.VerifierCouverture`. La boucle **doit** le rejeter en nommant
précisément ce qui manque — c'est la capacité de *vérification* d'un système
composable, qui empêche d'exécuter un plan à moitié formé.

In [9]:
// Plan incomplet — vérifier le rejet et l'admission

// PLAN INCOMPLET : x.md est decoupe en deux blocs, mais b1 n'a AUCUN traitement
// final. L'invariant de couverture doit rejeter le plan en nommant b1.
var planCasse = new Dictionary<string, EntreeRecette>(StringComparer.Ordinal)
{
    ["x.md#0"] = new EntreeRecette(new Decision("decouper", "blocs_markdown"),
        new[] { "x.md#0.b0", "x.md#0.b1" }),
    ["x.md#0.b0"] = new EntreeRecette(new Decision("compresser", "zlib"))
    // x.md#0.b1 : absent de la recette -> non couvert
};

// Capturer uniquement le rejet du vérificateur ; toute autre erreur remonte.
(bool Accepte, string Diagnostic) ExaminerCouverture(IEnumerable<string> racines,
    IReadOnlyDictionary<string, EntreeRecette> recette)
{
    try
    {
        Couverture.VerifierCouverture(racines, recette);
        return (true, string.Empty);
    }
    catch (CouvertureIncomplete e)
    {
        return (false, e.Message);
    }
}

var resultatPlanCasse = ExaminerCouverture(new[] { "x.md#0" }, planCasse);
// Diagnostic cité de la sortie Python, cellule 21.
const string referenceOrphelin = "1 segment(s) sans traitement final : ['x.md#0.b1']";
Egal("Vérification", "plan incomplet accepté", resultatPlanCasse.Accepte, false);
Egal("Vérification", "diagnostic exact du rejet", resultatPlanCasse.Diagnostic, referenceOrphelin);
Ecrire($"VÉRIFICATION : le plan incomplet est REJETE -> {resultatPlanCasse.Diagnostic}");

// CONTROLE POSITIF : le plan decouvert, lui, est complet -> l'invariant le laisse passer.
var resultatPlanDecouvert = ExaminerCouverture(racinesDecouverte.Select(seg => seg.Id), agent.Recette);
Egal("Vérification", "plan découvert accepté", resultatPlanDecouvert.Accepte, true);
Console.WriteLine("VÉRIFICATION : le plan decouvert est complet (aucun segment orphelin) -> laissé passer");

[CONFORME] Vérification / plan incomplet accepté : C# = False ; référence Python citée = False ; écart = 0


[CONFORME] Vérification / diagnostic exact du rejet : C# = 1 segment(s) sans traitement final : ['x.md#0.b1'] ; référence Python citée = 1 segment(s) sans traitement final : ['x.md#0.b1'] ; écart = 0


VÉRIFICATION : le plan incomplet est REJETE -> 1 segment(s) sans traitement final : ['x.md#0.b1']


[CONFORME] Vérification / plan découvert accepté : C# = True ; référence Python citée = True ; écart = 0


VÉRIFICATION : le plan decouvert est complet (aucun segment orphelin) -> laissé passer


### Lecture du résultat

Le même invariant **rejette** le plan à moitié formé (il nomme `x.md#0.b1`, l'enfant
orphelin) et **laisse passer** le plan découvert (tous les segments y ont un
traitement final). C'est la différence entre *avoir un plan* et *avoir un plan
exécutable* — et c'est ce qui autorise la composition : on ne branche un
mécanisme d'inférence que derrière une porte de vérification qui garantit que le
plan produit est complet.

## 8. La safety bound — arrêter une découverte divergente

La deuxième capacité de sécurité d'un système composable : **ne jamais tourner à
l'infini**. On branche un cerveau *pathologique* qui découpe toujours, sans
produire de traitement final — le parcours ne s'épuise jamais. Le squelette doit
l'**arrêter** à la borne d'appels, au lieu de laisser l'agent s'épuiser.

In [10]:
// CerveauDivergent — vérifier l'arrêt à la borne

// Ce cerveau découpe toujours, sans jamais proposer de traitement final.
sealed class CerveauDivergent : Cerveau
{
    public override Decision Inferer(Segment seg)
    {
        Appels++;
        return new Decision("decouper", "blocs_markdown");
    }
}

var divergent = new AgentTriage(new CerveauDivergent(), borne: 8);
try
{
    // texte SANS en-tete : blocs_markdown ne produit qu'un seul bloc = le segment
    // lui-meme, re-decoupe a l'identique -> le parcours ne s'epuise jamais.
    var boucle = new Segment("boucle.txt#0",
        Perception.Utf8.GetBytes("texte sans en-tete, a decouper a l'infini\n"), "texte");
    divergent.Decouvrir(new[] { boucle });
    throw new InvalidOperationException("La découverte divergente aurait dû être arrêtée à la borne.");
}
catch (BorneDecouverteAtteinte e)
{
    // Sortie Python citée, cellule 24 : deux compteurs et diagnostic exact.
    const string sectionBorne = "Borne de sûreté";
    const string referenceBorne = "decouverte interrompue a 8 inférences, 1 segment(s) en attente — safety bound.";
    Egal(sectionBorne, "inférences à l'arrêt", e.Inferences, 8);
    Egal(sectionBorne, "appels réels du cerveau", divergent.Cerveau.Appels, e.Inferences, "compteur à l'arrêt");
    Egal(sectionBorne, "segments en attente", e.EnAttente, 1);
    Egal(sectionBorne, "diagnostic exact", e.Message, referenceBorne);
    Ecrire($"SAFETY BOUND : la decouverte divergente est arretee -> {e.Message}");
}

[CONFORME] Borne de sûreté / inférences à l'arrêt : C# = 8 ; référence Python citée = 8 ; écart = 0


[CONFORME] Borne de sûreté / appels réels du cerveau : C# = 8 ; compteur à l'arrêt = 8 ; écart = 0


[CONFORME] Borne de sûreté / segments en attente : C# = 1 ; référence Python citée = 1 ; écart = 0


[CONFORME] Borne de sûreté / diagnostic exact : C# = decouverte interrompue a 8 inférences, 1 segment(s) en attente — safety bound. ; référence Python citée = decouverte interrompue a 8 inférences, 1 segment(s) en attente — safety bound. ; écart = 0


SAFETY BOUND : la decouverte divergente est arretee -> decouverte interrompue a 8 inférences, 1 segment(s) en attente — safety bound.


### Lecture du résultat

Le cerveau divergent découpe indéfiniment (un bloc sans en-tête se « découpe » en
un seul bloc identique) : sans borne, l'agent tournerait pour toujours. Le
squelette l'**arrête à 8 inférences** et signale l'échec de la découverte. C'est
la contrepartie de la composition : on accepte de brancher n'importe quel
mécanisme d'inférence *parce que* le squelette porte les deux garde-fous — la
**vérification** de complétude (section 7) et la **safety bound** (ici). Sans
eux, « composable » ne serait que « instable ».

## 9. Le second cerveau — le LLM, point de composition

On montre maintenant concrètement le **point de composition** : le *même*
`AgentTriage` accepte un cerveau **LLM** (Semantic Kernel, tel que
Compression-07b l'emploie) aussi bien que le cerveau déterministe, parce qu'il ne
voit que le contrat de la classe abstraite `Cerveau`. Le cerveau LLM est **déclaré, pas invoqué** ici —
le carnet reste reproductible sans clé API — et c'est exactement le propos : le
cerveau est une *pièce* amovible, pas le squelette.

In [11]:
// CerveauLLM — vérifier l'interchangeabilité

// Le point de composition est le même contrat Cerveau. Dans Compression-07b,
// l'adaptateur lit une décision compresser/decouper via Semantic Kernel ;
// ici il est seulement déclaré, sans dépendance ni appel externe.
sealed class CerveauLLM : Cerveau
{
    public override Decision Inferer(Segment seg) =>
        throw new InvalidOperationException("CerveauLLM.Inferer n'est pas câblé dans ce carnet.");
}

// Le point de composition, concret : le meme squelette accepte les deux cerveaux,
// car son constructeur ne dépend que du type abstrait Cerveau.
var cerveauxComposition = new (string Nom, Cerveau Cerveau)[]
{
    ("deterministe", new CerveauDeterministe()),
    ("LLM", new CerveauLLM())
};
foreach (var (nom, cerveau) in cerveauxComposition)
{
    var agentCompose = new AgentTriage(cerveau);
    bool accepte = object.ReferenceEquals(agentCompose.Cerveau, cerveau);
    // Les deux verdicts True sont cités de la sortie Python, cellule 27.
    Egal("Composition", $"cerveau {nom} accepté", accepte, true);
    Ecrire($"   AgentTriage accepte un cerveau {nom,-14} : {accepte}");
}
Console.WriteLine("\nla boucle est agnostique du mecanisme d'inference : c'est la composition.");
Console.WriteLine("le resultat experimental ci-dessus a ete produit par le cerveau determine ;");
Console.WriteLine("brancher CerveauLLM changerait le cout, pas le squelette ni les garde-fous.");

[CONFORME] Composition / cerveau deterministe accepté : C# = True ; référence Python citée = True ; écart = 0


   AgentTriage accepte un cerveau deterministe   : True


[CONFORME] Composition / cerveau LLM accepté : C# = True ; référence Python citée = True ; écart = 0


   AgentTriage accepte un cerveau LLM            : True



la boucle est agnostique du mecanisme d'inference : c'est la composition.


le resultat experimental ci-dessus a ete produit par le cerveau determine ;


brancher CerveauLLM changerait le cout, pas le squelette ni les garde-fous.


### Lecture du résultat

Le même squelette accepte les deux cerveaux — le test `ReferenceEquals(agentCompose.Cerveau, cerveau)`
passe pour les deux, et c'est tout ce que la boucle exige. La composition est
réelle : **brancher un LLM change le coût d'inférence, pas le squelette, pas la
vérification, pas la safety bound.** Le résultat expérimental de ce carnet a été
produit par le cerveau déterministe (reproductible, sans clé) ; en production on
y brancherait `CerveauLLM`, et les sections 6 à 8 garderaient leur sens à
l'identique.

## 10. Exercices

Trois exercices, un par capacité du squelette : étendre la **perception**, ajouter
une **opération inversible**, et exploiter le **point de composition**.

### Exercice 1 — étendre la perception

Le corpus ne détecte que le CSV (virgules). Ajoutez la détection du **TSV**
(colonnes séparées par des tabulateurs) dans `Perception.StructureInterne`, puis une
signature `"tsv"` dans `Perception.Signature()` (avant `"csv"`).

In [12]:
// Illustration 1 — Exercice : détection du TSV
Dictionary<string, bool> StructureTsv(Segment seg)
{
    // TODO étudiant : renvoyer une signature de structure qui détecte le TSV
    // (colonnes séparées par des tabulateurs).
    // Le dictionnaire doit contenir au moins la clé "tsv" (bool).
    return new Dictionary<string, bool>(StringComparer.Ordinal); // TODO étudiant
}

if (StructureTsv(segments["ventes.csv#0"]).Count == 0)
    Console.WriteLine("Exercice 1 a completer : detection du TSV dans la perception");

Exercice 1 a completer : detection du TSV dans la perception


### Exercice 2 — une opération inversible

Écrivez un codec **RLE** (run-length : `(octet, compte)`) et enregistrez-le dans
`OperationsTriage.Codecs`. Le contrat d'inversibilité doit tenir : `DecompresserRle(CompresserRle(d)).SequenceEqual(d)`.

In [13]:
// Illustration 2 — Exercice : codec RLE inversible
byte[] CompresserRle(byte[] donnees)
{
    // TODO étudiant : produire la représentation RLE en paires (octet, compte).
    return Array.Empty<byte>(); // TODO étudiant
}

byte[] DecompresserRle(byte[] donnees)
{
    // TODO étudiant : reconstruire les octets d'origine depuis les paires.
    return Array.Empty<byte>(); // TODO étudiant
}

if (CompresserRle(Perception.Utf8.GetBytes("aaab")).Length == 0)
    Console.WriteLine("Exercice 2 a completer : codec RLE inversible");

Exercice 2 a completer : codec RLE inversible


### Exercice 3 — le point de composition

Écrivez un `CerveauHybride` qui **route** : les petits segments (moins d'un seuil
d'octets) sont confiés au cerveau déterministe, les gros renvoient
une `Decision` dont l'opération est `"escalade"` (à traiter par un LLM en production). Comptez les appels
comme les autres cerveaux.

In [14]:
// Illustration 3 — Exercice : cerveau hybride
sealed class CerveauHybride : Cerveau
{
    public Cerveau Petit { get; }
    public int Seuil { get; }

    public CerveauHybride(Cerveau petit, int seuil = 128)
    {
        Petit = petit;
        Seuil = seuil;
    }

    public override Decision Inferer(Segment seg)
    {
        // TODO étudiant : router selon la taille du segment et le seuil.
        return new Decision(string.Empty, string.Empty); // TODO étudiant
    }
}

// Le carnet doit rester exécutable : ce squelette renvoie une décision neutre tant qu'il est incomplet.
Console.WriteLine("Exercice 3 a completer : cerveau hybride (routing petit/gros)");

Exercice 3 a completer : cerveau hybride (routing petit/gros)


In [15]:
// Attestation de parité — agrège les écarts enregistrés par les assertions des sections
// précédentes. Elle est HORS de l'exercice 3 : les exercices restent indépendants d'elle,
// et le squelette laissé ci-dessus n'est pas exécuté par cette attestation.
{
    var sectionsAttendues = new[]
    {
        "Perception", "Aperçu", "Opérations inversibles", "Découverte",
        "Rejeu", "Observation", "Vérification", "Borne de sûreté", "Composition"
    };
    var mesuresParSection = comparaisons.ToLookup(mesure => mesure.Section, StringComparer.Ordinal);
    if (!new HashSet<string>(sectionsAttendues, StringComparer.Ordinal)
            .SetEquals(mesuresParSection.Select(groupe => groupe.Key)))
        throw new InvalidOperationException("Le journal des comparaisons ne couvre pas les sections attendues.");

    Console.WriteLine("\nATTESTATION DE PARITÉ — écarts mesurés par section");
    Ecrire($"   Tolérance : {ToleranceEgalite} différence ; le codec Brotli est comparé à LZMA par correspondance déclarée.");
    foreach (string section in sectionsAttendues)
    {
        var mesures = mesuresParSection[section];
        int controles = mesures.Count();
        int ecartMax = mesures.Max(mesure => mesure.Ecart);
        int ecarts = mesures.Count(mesure => mesure.Ecart > ToleranceEgalite);
        string verdict = ecarts == 0 ? "CONFORME" : "ÉCART";
        Ecrire($"   [{verdict}] {section} : {controles} contrôles, {ecarts} écarts, écart maximal {ecartMax}.");
        if (ecarts != 0)
            throw new InvalidOperationException($"Parité hors tolérance : {section}.");
    }
}


ATTESTATION DE PARITÉ — écarts mesurés par section


   Tolérance : 0 différence ; le codec Brotli est comparé à LZMA par correspondance déclarée.


   [CONFORME] Perception : 7 contrôles, 0 écarts, écart maximal 0.


   [CONFORME] Aperçu : 10 contrôles, 0 écarts, écart maximal 0.


   [CONFORME] Opérations inversibles : 27 contrôles, 0 écarts, écart maximal 0.


   [CONFORME] Découverte : 12 contrôles, 0 écarts, écart maximal 0.


   [CONFORME] Rejeu : 4 contrôles, 0 écarts, écart maximal 0.


   [CONFORME] Observation : 5 contrôles, 0 écarts, écart maximal 0.


   [CONFORME] Vérification : 3 contrôles, 0 écarts, écart maximal 0.


   [CONFORME] Borne de sûreté : 4 contrôles, 0 écarts, écart maximal 0.


   [CONFORME] Composition : 2 contrôles, 0 écarts, écart maximal 0.


## 11. À retenir

- **Un agent composable = un squelette + des pièces.** La boucle
  percevoir→représenter→inférer→décider→vérifier→exécuter→observer est le
  squelette ; l'inférence est une *pièce* derrière une interface.
- **L'inférence est un point de composition.** Déterministe, LLM, hybride : le
  squelette, la vérification et la safety bound ne changent pas — seul le **coût**
  d'inférence change.
- **La connaissance s'amortit.** Découvrir une politique `forme → décision` coûte des
  inférences une fois ; la rejouer en coûte zéro. La connaissance est un actif.
- **Composer exige des garde-fous.** Un mécanisme d'inférence ne se branche que
  derrière une **vérification de complétude** (l'invariant de couverture rejette un
  plan à moitié formé) et une **safety bound** (on arrête une découverte
  divergente).
- **Les primitives se réutilisent.** La représentation, l'invariant et la boucle
  bornée viennent de Compression-07b ; ce carnet les *compose*, il ne les réinvente
  pas.

## Vérifiez votre compréhension

1. Pourquoi le stade **Vérifier** est-il placé *dans* la découverte, et non après
   l'exécution ? *(Réponse repliée ci-dessous.)*
2. Que mesure exactement le compteur `AppelDecouverte`, et pourquoi le delta du
   compteur `Cerveau.Appels` est-il de 0 pendant `Rejouer` ?
3. Quel est le risque précis que la **safety bound** empêche, et qu'arriverait-il
   sans elle sur un cerveau divergent ?

<details><summary>Réponse 1</summary>
Parce qu'un plan incomplet ne doit **jamais** être exécuté : la vérification est
la *porte d'admission* du plan. Si elle était après l'exécution, on aurait déjà
transformé des données avec un plan à moitié formé — trop tard pour rejeter.
</details>

<details><summary>Réponse 2</summary>
Il compte les **consultations du cerveau** (les véritables inférences). Pendant
`Rejouer`, le catalogue `forme → décision` est *consulté* (une table), pas le
cerveau — d'où 0. C'est tout l'amortissement.
</details>

<details><summary>Réponse 3</summary>
Un cerveau qui découpe toujours sans produire de traitement final laisse la
file non vide : sans borne, l'agent **tourne à l'infini** (temps et coût sans
fin). La safety bound arrête la découverte et signale l'échec.
</details>

## Pour aller plus loin

- [`Compression-07b`](../../InformationCalcul/Compression/Compression-07b-Triage-LLM-Semantic-Kernel-Python.ipynb) —
  l'organe dont ce carnet réutilise les primitives (le triage LLM complet, le
  rejeu sans LLM, la réfutation « le triage bat-il l'indifférencié ? »).

> Ce carnet est le jumeau C# de la [version Python](AgentSystems-01-Perception-Vers-Action-Python.ipynb).